# NBA Game Prediction Pipeline

**Overview**

End-to-end machine learning pipeline for NBA game outcome prediction using leakage-free chronological feature engineering and logistic regression.

## Stage 1: Load and Prepare Data

In [ ]:
import pandas as pd

df = pd.read_csv("../data/nba_games.csv")

df["gameDate"] = pd.to_datetime(df["gameDateTimeEst"])

df = df[df["gameDate"] >= "2000-01-01"].copy()

df = df.sort_values("gameDate").reset_index(drop=True)

df["HOME_WIN"] = (df["homeScore"] > df["awayScore"]).astype(int)

df = df[
    [
        "gameId",
        "gameDate",
        "hometeamName",
        "awayteamName",
        "homeScore",
        "awayScore",
        "HOME_WIN"
    ]
].copy()

df.head()
df.info()

## Stage 2: Build Chronological Team Timeline

In [ ]:
home = df[[
    "gameId", "gameDate", "hometeamName", "homeScore", "awayScore"
]].copy()

home.columns = ["gameId", "gameDate", "team", "points_scored", "points_allowed"]
home["is_home"] = 1
home["win"] = (home["points_scored"] > home["points_allowed"]).astype(int)

away = df[[
    "gameId", "gameDate", "awayteamName", "awayScore", "homeScore"
]].copy()

away.columns = ["gameId", "gameDate", "team", "points_scored", "points_allowed"]
away["is_home"] = 0
away["win"] = (away["points_scored"] > away["points_allowed"]).astype(int)

team_timeline = pd.concat([home, away], ignore_index=True)

team_timeline = team_timeline.sort_values(["team", "gameDate"]).reset_index(drop=True)

team_timeline["point_diff"] = (
    team_timeline["points_scored"] - team_timeline["points_allowed"]
)

team_timeline.head()

## Stage 3: Leakage-Free Feature Engineering

In [ ]:
team_timeline = team_timeline.sort_values(["team", "gameDate"]).copy()

team_timeline["win_5"] = (
    team_timeline.groupby("team")["win"]
    .transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))
)

team_timeline["win_10"] = (
    team_timeline.groupby("team")["win"]
    .transform(lambda x: x.rolling(10, min_periods=1).mean().shift(1))
)

team_timeline["pd_5"] = (
    team_timeline.groupby("team")["point_diff"]
    .transform(lambda x: x.rolling(5, min_periods=1).mean().shift(1))
)

team_timeline["pd_10"] = (
    team_timeline.groupby("team")["point_diff"]
    .transform(lambda x: x.rolling(10, min_periods=1).mean().shift(1))
)

team_timeline.head(10)

## Stage 4: Build Machine Learning Dataset

In [ ]:
ml_base = df[[
    "gameId",
    "gameDate",
    "hometeamName",
    "awayteamName",
    "homeScore",
    "awayScore",
    "HOME_WIN"
]].copy()

features = team_timeline[[
    "gameId",
    "team",
    "win_5",
    "win_10",
    "pd_5",
    "pd_10"
]].copy()

home_features = features.rename(columns={
    "team": "hometeamName",
    "win_5": "home_win_5",
    "win_10": "home_win_10",
    "pd_5": "home_pd_5",
    "pd_10": "home_pd_10"
})

ml_base = ml_base.merge(home_features, on=["gameId", "hometeamName"], how="left")

away_features = features.rename(columns={
    "team": "awayteamName",
    "win_5": "away_win_5",
    "win_10": "away_win_10",
    "pd_5": "away_pd_5",
    "pd_10": "away_pd_10"
})

ml_base = ml_base.merge(away_features, on=["gameId", "awayteamName"], how="left")

ml_base = ml_base.dropna().reset_index(drop=True)

ml_base.head()

## Stage 5: Chronological Train/Test Split

In [ ]:
ml_base = ml_base.sort_values("gameDate").reset_index(drop=True)

feature_cols = [
    "home_win_5","away_win_5",
    "home_win_10","away_win_10",
    "home_pd_5","away_pd_5",
    "home_pd_10","away_pd_10"
]

X = ml_base[feature_cols]
y = ml_base["HOME_WIN"]

split_index = int(len(ml_base) * 0.80)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(len(X_train), len(X_test))
print(ml_base.iloc[split_index]["gameDate"])

## Stage 6: Train Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)

model.fit(X_train, y_train)

model.coef_, model.intercept_

## Stage 7: Evaluate Model

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print(confusion_matrix(y_test, y_pred))

print(classification_report(y_test, y_pred))

## Stage 8: Predict New Games

In [ ]:
current_teams = [
    "Hawks","Celtics","Nets","Hornets","Bulls","Cavaliers",
    "Mavericks","Nuggets","Pistons","Warriors","Rockets",
    "Pacers","Clippers","Lakers","Grizzlies","Heat","Bucks",
    "Timberwolves","Pelicans","Knicks","Thunder","Magic",
    "76ers","Suns","Trail Blazers","Kings","Spurs",
    "Raptors","Jazz","Wizards"
]

def get_team_latest(team_name, team_timeline):
    temp = team_timeline[team_timeline["team"] == team_name].sort_values("gameDate")
    if temp.empty:
        raise ValueError(f"No data found for team: {team_name}")
    return temp.iloc[-1]


def predict_matchup(home_team, away_team):

    if home_team == away_team:
        return {"error": "Invalid matchup: same team cannot play itself"}

    home = get_team_latest(home_team, team_timeline)
    away = get_team_latest(away_team, team_timeline)

    input_data = pd.DataFrame([{
        "home_win_5": home["win_5"],
        "home_win_10": home["win_10"],
        "home_pd_5": home["pd_5"],
        "home_pd_10": home["pd_10"],
        "away_win_5": away["win_5"],
        "away_win_10": away["win_10"],
        "away_pd_5": away["pd_5"],
        "away_pd_10": away["pd_10"]
    }])

    input_data = input_data[X_train.columns]

    probs = model.predict_proba(input_data)[0]

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_prob": round(float(probs[1]), 3),
        "away_prob": round(float(probs[0]), 3),
        "predicted_winner": home_team if probs[1] > probs[0] else away_team
    }

In [ ]:
# test

games = [
    ("Lakers", "Celtics"),
    ("Heat", "Bucks"),
    ("Lakers", "Lakers"),
    ("Spurs","Knicks"),
    ("Knicks","Spurs")
]

for home, away in games:
    print(predict_matchup(home, away))